## Exploring predictions

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import sys
import gc
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import silence_tensorflow

import experiment_settings
import build_model
import build_data
import plots
import methods
import save_files
import read_landsat

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# GET SETTINGS
EXP_NAME = "exp1"
settings = experiment_settings.get_settings(EXP_NAME)

directory_paths = methods.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figures_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]

In [ ]:
# GET THE DATA
import read_landsat
import methods
imp.reload(build_data)
imp.reload(read_landsat)
imp.reload(methods)

settings["batch_size"] = 256
settings["mode"] = "inference"
settings["inference_region"] = (-7.9, -5.1, 105.1, 107.9)

for year in (2022,):#np.arange(2019,2023):

    print(' --- ' + str(year) + '---')
    settings["inference_years"] = (year, )
    filenames_list = []
        
    min_latfile, max_latfile, min_lonfile, max_lonfile = read_landsat.get_landsat_bounds(settings, region=settings["inference_region"])

    for latfile in np.arange(min_latfile + settings["tile_len_deg"], max_latfile + settings["tile_len_deg"], settings["tile_len_deg"]):
        for lonfile in np.arange(min_lonfile, max_lonfile, settings["tile_len_deg"]):

            settings["tile"] = (latfile - settings["tile_len_deg"], latfile, lonfile, lonfile + settings["tile_len_deg"])

            # GET THE SAMPLE TAGS
            tags_inf, __ = build_data.get_tags(settings)
            tfds_inf = build_data.build_tf_dataset(settings, tags_inf, settings["batch_size"])
            tfds_inf = tfds_inf.prefetch(tf.data.AUTOTUNE)
            print(f"file = {tags_inf[-1][0]}")

            # LOAD THE MODEL AND MAKE PREDICTIONS
            # TODO: move outside of the loop
            tf.keras.backend.clear_session()
            model = build_model.build_model(settings, 
                                            input_shape=np.shape(next(tfds_inf.as_numpy_iterator())[0])[1:])

            checkpoint_dir = SAVE_MODEL_DIRECTORY + settings["exp_name"] + '/'
            model.load_weights(tf.train.latest_checkpoint(checkpoint_dir)).expect_partial()

            hfi_predict = model.predict(tfds_inf, verbose=1)[:,0]
            __ = gc.collect()

            # SAVE THE PREDICTIONS AS A TIF
            predictions_filename = settings["exp_name"] + "_predictions_" + tags_inf[-1][0]
            hfi_predict, hfi_labels, lat0, lat1, lon0, lon1 = save_files.save_predictions_tif(settings, tags_inf, hfi_predict, predictions_filename)
            filenames_list.append(predictions_filename + ".tif")

            # PLOT THE RESULTS
            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plots.plot_hfi_tile(hfi_predict, [lon0, lon1, lat1, lat0])
            plt.title('mlHFI Predictions for ' + str(settings["inference_years"][0]))
            plt.clim(0,100)

            plt.subplot(1,2,2)
            plots.plot_hfi_tile(hfi_labels, [lon0, lon1, lat1, lat0])
            plt.title('HFI Labels for ' + str(settings["inference_years"][0]))
            plt.clim(0,100)

            plt.savefig(FIGURE_DIRECTORY + predictions_filename + ".png")
            plt.show()
            # plt.close()

In [ ]:
# import tifftools
# # import rasterio
# # from rasterio import merge

# with rasterio.open(f1) as tif1:
#     with rasterio.open(f2) as tif2:
#         x, transform = merge.merge(filenames)
        

# plt.imshow(x[0,:,:])